# Agent Observer (Phase 6: MQTT Aggregator + Publisher)

This notebook subscribes to simulation state/event topics and publishes observer metrics.

Phase 6 scope:
- Subscribe: `.../sim/tick`, `.../sim/sheep/state`, `.../sim/wolf/state`, `.../sim/grass/state`, `.../sim/events`, `.../sim/controller/commands`
- Publish: `.../sim/observer/metrics`

In [1]:
# Cell purpose: import dependencies and connect to MQTT using project config.
from datetime import datetime, timezone
import json
import time

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher

config = load_config()
print(f"Loaded config. Primary MQTT profile: {config.mqtt.host}:{config.mqtt.port}")

connector = MqttConnector(config.mqtt, client_id_suffix="agent-observer-phase6")
connector.connect()
connected = connector.wait_for_connection(timeout=5)
print(f"MQTT connected: {connected}")

publisher = MqttPublisher(connector)
print("STARTUP_OK: agent_observer connected and ready")

Loaded config. Primary MQTT profile: 127.0.0.1:1883
MQTT connected: True
STARTUP_OK: agent_observer connected and ready


In [2]:
# Cell purpose: define topic names and observer aggregation state.
base_topic = config.mqtt.base_topic.rstrip("/")
tick_topic = f"{base_topic}/sim/tick"
sheep_state_topic = f"{base_topic}/sim/sheep/state"
wolf_state_topic = f"{base_topic}/sim/wolf/state"
grass_state_topic = f"{base_topic}/sim/grass/state"
events_topic = f"{base_topic}/sim/events"
controller_commands_topic = f"{base_topic}/sim/controller/commands"
observer_metrics_topic = f"{base_topic}/sim/observer/metrics"

simulation_cfg = config.simulation
grid_width = simulation_cfg.grid_width if simulation_cfg is not None else 10
grid_height = simulation_cfg.grid_height if simulation_cfg is not None else 10
total_cells = grid_width * grid_height

latest_tick = 0
latest_sheep_count = simulation_cfg.initial_sheep if simulation_cfg is not None else 30
latest_wolf_count = simulation_cfg.initial_wolves if simulation_cfg is not None else 8
latest_sheep_avg_energy = 0.0
latest_grass_grown_cells = int(total_cells * ((simulation_cfg.initial_grass_coverage_pct if simulation_cfg is not None else 70) / 100.0))
latest_grass_coverage_pct = round((latest_grass_grown_cells / total_cells) * 100.0, 2) if total_cells > 0 else 0.0
latest_controller_command = "baseline"

event_counts_by_tick = {}

print("Subscribe topics:")
print(f"- {tick_topic}")
print(f"- {sheep_state_topic}")
print(f"- {wolf_state_topic}")
print(f"- {grass_state_topic}")
print(f"- {events_topic}")
print(f"- {controller_commands_topic}")
print("Publish topics:")
print(f"- {observer_metrics_topic}")

Subscribe topics:
- simulated-city/sim/tick
- simulated-city/sim/sheep/state
- simulated-city/sim/wolf/state
- simulated-city/sim/grass/state
- simulated-city/sim/events
- simulated-city/sim/controller/commands
Publish topics:
- simulated-city/sim/observer/metrics


In [3]:
# Cell purpose: aggregate incoming messages and publish observer metrics every tick.
def _tick_events_entry(tick):
    if tick not in event_counts_by_tick:
        event_counts_by_tick[tick] = {
            "births": 0,
            "deaths": 0,
            "predation": 0,
            "sheep_ate_grass": 0,
            "wolf_births": 0,
            "wolf_deaths": 0,
        }
    return event_counts_by_tick[tick]


def on_message(client, userdata, msg):
    global latest_tick, latest_sheep_count, latest_wolf_count
    global latest_sheep_avg_energy, latest_grass_grown_cells, latest_grass_coverage_pct
    global latest_controller_command

    try:
        payload = json.loads(msg.payload.decode("utf-8"))
    except Exception:
        print(f"Skipping non-JSON payload on topic {msg.topic}")
        return

    if msg.topic == sheep_state_topic:
        latest_sheep_count = int(payload.get("sheep_count", latest_sheep_count))
        latest_sheep_avg_energy = float(payload.get("average_energy", latest_sheep_avg_energy))
        return

    if msg.topic == wolf_state_topic:
        latest_wolf_count = int(payload.get("wolf_count", latest_wolf_count))
        return

    if msg.topic == grass_state_topic:
        latest_grass_grown_cells = int(payload.get("grass_grown_cells", latest_grass_grown_cells))
        latest_grass_coverage_pct = float(payload.get("grass_coverage_pct", latest_grass_coverage_pct))
        return

    if msg.topic == controller_commands_topic:
        reasons = payload.get("reason", ["baseline"])
        if isinstance(reasons, list):
            latest_controller_command = ",".join(str(reason) for reason in reasons)
        else:
            latest_controller_command = str(reasons)
        return

    if msg.topic == events_topic:
        tick = int(payload.get("tick", latest_tick))
        entry = _tick_events_entry(tick)
        entry["births"] += int(payload.get("births", 0))
        entry["deaths"] += int(payload.get("deaths", 0))
        entry["predation"] += int(payload.get("estimated_predation", 0))
        entry["sheep_ate_grass"] += int(payload.get("sheep_ate_grass", 0))
        entry["wolf_births"] += int(payload.get("wolf_births", 0))
        entry["wolf_deaths"] += int(payload.get("wolf_deaths", 0))
        return

    if msg.topic == tick_topic:
        tick = int(payload.get("tick", latest_tick + 1))
        latest_tick = tick

        event_entry = _tick_events_entry(tick)
        occupancy_ratio = round(min(1.0, (latest_sheep_count + latest_wolf_count) / total_cells), 4) if total_cells > 0 else 0.0

        metrics_payload = {
            "tick": tick,
            "population": {
                "sheep": latest_sheep_count,
                "wolves": latest_wolf_count,
            },
            "energy": {
                "sheep_average": round(latest_sheep_avg_energy, 3),
            },
            "grass": {
                "grown_cells": latest_grass_grown_cells,
                "coverage_pct": round(latest_grass_coverage_pct, 2),
            },
            "events": event_entry,
            "occupancy": {
                "estimated_ratio": occupancy_ratio,
                "grid_width": grid_width,
                "grid_height": grid_height,
                "method": "(sheep+wolves)/cells",
            },
            "controller": {
                "latest_reason": latest_controller_command,
            },
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        publisher.publish_json(observer_metrics_topic, json.dumps(metrics_payload), qos=0)
        print(
            f"tick={tick} sheep={latest_sheep_count} wolves={latest_wolf_count} "
            f"grass={round(latest_grass_coverage_pct, 2)}% occupancy={occupancy_ratio} "
            f"events={event_entry} command={latest_controller_command} | published"
        )

        # Drop old tick event accumulators to keep memory bounded.
        old_ticks = [value for value in event_counts_by_tick.keys() if value < tick - 2]
        for old_tick in old_ticks:
            event_counts_by_tick.pop(old_tick, None)


connector.client.on_message = on_message
connector.client.subscribe(tick_topic, qos=0)
connector.client.subscribe(sheep_state_topic, qos=0)
connector.client.subscribe(wolf_state_topic, qos=0)
connector.client.subscribe(grass_state_topic, qos=0)
connector.client.subscribe(events_topic, qos=0)
connector.client.subscribe(controller_commands_topic, qos=0)
print("Subscriptions active.")

Subscriptions active.


In [ ]:
# Cell purpose: keep this notebook alive to process incoming MQTT messages.
print("Observer agent loop running. Interrupt the cell to stop.")
try:
    while True:
        time.sleep(0.2)
except KeyboardInterrupt:
    print("Stopping observer agent loop.")
finally:
    connector.disconnect()
    print("MQTT disconnected.")

Observer agent loop running. Interrupt the cell to stop.


tick=64 sheep=30 wolves=8 grass=70.0% occupancy=0.38 events={'births': 0, 'deaths': 0, 'predation': 0, 'sheep_ate_grass': 0, 'wolf_births': 0, 'wolf_deaths': 0} command=baseline | published
tick=65 sheep=68 wolves=9 grass=39.0% occupancy=0.77 events={'births': 0, 'deaths': 0, 'predation': 0, 'sheep_ate_grass': 0, 'wolf_births': 0, 'wolf_deaths': 0} command=baseline | published
tick=66 sheep=68 wolves=9 grass=35.0% occupancy=0.77 events={'births': 0, 'deaths': 0, 'predation': 0, 'sheep_ate_grass': 0, 'wolf_births': 0, 'wolf_deaths': 0} command=baseline | published
tick=67 sheep=69 wolves=9 grass=39.0% occupancy=0.78 events={'births': 0, 'deaths': 0, 'predation': 0, 'sheep_ate_grass': 0, 'wolf_births': 0, 'wolf_deaths': 0} command=baseline | published
tick=68 sheep=68 wolves=9 grass=38.0% occupancy=0.77 events={'births': 0, 'deaths': 0, 'predation': 0, 'sheep_ate_grass': 0, 'wolf_births': 0, 'wolf_deaths': 0} command=baseline | published
tick=69 sheep=65 wolves=9 grass=35.0% occupancy=0.